# Adult Census Income - Explorative Data Analysis (EDA)

## Setup

In [ ]:
import json
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path
from ucimlrepo import fetch_ucirepo
from scipy.stats import contingency

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
RAW_PATH = Path("../data/raw/adult_raw.csv")

if RAW_PATH.exists():
    df_raw = pd.read_csv(RAW_PATH)    
else:
    adult = fetch_ucirepo(id=2)

    df_raw = adult.data.original.copy()

    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_raw.to_csv(RAW_PATH, index=False)    

df = df_raw.copy()

## Dataset Overview

In [ ]:
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print()


summary = pd.DataFrame({
    "dtype":          df.dtypes,
    "unique_values":  df.nunique(),
    "sample_value":   df.iloc[0],
})

summary

## Missing Values

In [ ]:
df.replace("?", pd.NA, inplace=True)

missing = pd.DataFrame({
    "missing_count":   df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df) * 100).round(2)
}).query("missing_count > 0").sort_values("missing_percent", ascending=False)

missing

In [ ]:
ax = missing["missing_percent"].plot(kind="barh", figsize=(7, 3), color="steelblue")
ax.set_xlabel("Missing (%)")
ax.set_title("Missing Values by Feature")
plt.tight_layout()
plt.show()

## Feature Types

In [ ]:
numerical_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

target = metadata["target_col"]
if isinstance(target, list):
    target = target[0]
categorical_cols = [c for c in categorical_cols if c != target]

print(f"Numerical features ({len(numerical_cols)}):   {numerical_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Target: {target}")

In [ ]:
feature_types = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.values,
    "kind": ["target" if c == target 
             else "numerical" if c in numerical_cols 
             else "categorical" 
             for c in df.columns]
})

feature_types

In [ ]:
print(df[["education", "education-num"]].drop_duplicates().sort_values("education-num"))

Note: 'education-num' and 'education' appear to encode the same information 
in two different representations. This redundancy should be considered during preprocessing.

## Target Distribution

In [ ]:
df[target] = df[target].str.strip().str.replace(".", "", regex=False)
print(df[target].value_counts())

In [ ]:
target_counts = df[target].value_counts()
target_pct = df[target].value_counts(normalize=True).mul(100).round(2)

target_dist = pd.DataFrame({
    "count": target_counts,
    "percent": target_pct
})

print(target_dist)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))


target_counts.plot(kind="bar", ax=axes[0], color=["steelblue", "coral"], 
                   edgecolor="white", rot=0)
axes[0].set_title("Anzahl Beobachtungen pro Klasse")
axes[0].set_xlabel("Einkommen")
axes[0].set_ylabel("Anzahl")
for bar in axes[0].patches:
    axes[0].annotate(f"{int(bar.get_height()):,}",
                     (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                     ha="center", va="bottom", fontsize=10)


axes[1].pie(target_counts, labels=target_counts.index, autopct="%1.1f%%",
            colors=["steelblue", "coral"], startangle=90)
axes[1].set_title("Klassenanteile")

plt.tight_layout()
plt.show()

## Numerical Feature Analysis

In [ ]:
df[numerical_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue", bins=40)
    axes[i].set_title(col)
    axes[i].set_xlabel("")

plt.suptitle("Verteilungen numerischer Merkmale", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.boxplot(x=df[col], ax=axes[i], color="steelblue")
    axes[i].set_title(col)

plt.suptitle("Boxplots numerischer Merkmale", y=1.02)
plt.tight_layout()
plt.show()

Observations:
- age: slightly right-skewed
- fnlwgt: high variance, census weight, candidate for dropping in preprocessing
- education-num: 16 discrete values, better treated as ordinal
- capital-gain / capital-loss: heavily zero-inflated, challenging for generative models
- hours-per-week: peak at 40, long right tail

## Categorical Feature Analysis

In [ ]:
for col in categorical_cols:
    print(f"\n{col} ({df[col].nunique()} unique values):")
    print(df[col].value_counts())

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    order = df[col].value_counts().index
    sns.countplot(y=df[col], ax=axes[i], order=order, color="steelblue")
    axes[i].set_title(col)
    axes[i].set_xlabel("Count")
    axes[i].set_ylabel("")


for j in range(len(categorical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Categorical Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Note: 'native-country-grouped' is created for visualization purposes only.
# It is not intended for use in preprocessing or modeling.
threshold = 100
country_counts = df["native-country"].value_counts()
rare_countries = country_counts[country_counts < threshold].index

df["native-country-grouped"] = df["native-country"].apply(
    lambda x: "Other" if x in rare_countries else x
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
order = df["native-country-grouped"].value_counts().index
sns.countplot(y=df["native-country-grouped"], order=order, color="steelblue", ax=ax)
ax.set_title("Native Country Distribution")
ax.set_xlabel("Count")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

Observations:

- race: heavily imbalanced, risk of generative model underlearning minority patterns
- native-country: dominated by United-States, many rare categories — poor synthesis candidate
- workclass: dominated by Private, rare categories may be lost in synthesis
- sex: ~2:1 male/female ratio, less critical but worth monitoring

Potential fairness implications for minority groups in synthetic data evaluation

## Feature Relationships

This section looks at how features relate to the target variable

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.boxplot(x=df[target], y=df[col], ax=axes[i],
                hue=df[target], legend=False,
                palette={"<=50K": "steelblue", ">50K": "coral"})
    axes[i].set_title(col)
    axes[i].set_xlabel("")

plt.suptitle("Numerische Merkmale vs. Zielmerkmal", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    prop = (df.groupby(col)[target]
              .value_counts(normalize=True)
              .mul(100)
              .rename("percent")
              .reset_index())
    
    pivot = prop.pivot(index=col, columns=target, values="percent").fillna(0)
    pivot = pivot.loc[df[col].value_counts().index]  
    
    pivot.plot(kind="barh", stacked=True, ax=axes[i],
               color=["steelblue", "coral"], legend=(i == 0))
    axes[i].set_title(col)
    axes[i].set_xlabel("Percent")
    axes[i].set_ylabel("")

for j in range(len(categorical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Categorical Features vs Target (Proportions)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

prop = (df.groupby("native-country-grouped")[target]
          .value_counts(normalize=True)
          .mul(100)
          .rename("percent")
          .reset_index())

pivot = prop.pivot(index="native-country-grouped", columns=target, values="percent").fillna(0)
pivot = pivot.loc[df["native-country-grouped"].value_counts().index]

pivot.plot(kind="barh", stacked=True, ax=ax,
           color=["steelblue", "coral"], legend=True)
ax.set_title("Native Country (Grouped) vs Target (Proportions)")
ax.set_xlabel("Percent")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

Note: The category 'South' in native-country is ambiguous. It is unclear 
whether this refers to a specific country or region. This appears to be 
an artifact of the original UCI dataset encoding and cannot be resolved 
without access to the original census documentation.

## Correlations

In [ ]:
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    return contingency.association(confusion_matrix, method="cramer")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

corr_matrix = df[numerical_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, ax=ax)
ax.set_title("Numerical Feature Correlations")
plt.tight_layout()
plt.show()

In [ ]:
cat_cols_with_target = categorical_cols + [target]

cramers_matrix = pd.DataFrame(index=cat_cols_with_target,
                               columns=cat_cols_with_target,
                               dtype=float)

for col1 in cat_cols_with_target:
    for col2 in cat_cols_with_target:
        cramers_matrix.loc[col1, col2] = cramers_v(df[col1].astype(str),
                                                     df[col2].astype(str))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cramers_matrix.astype(float), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, square=True, ax=ax)
ax.set_title("Kategoriale Merkmalsassoziationen (Cramér's V)")
plt.tight_layout()
plt.show()

Key correlations:

- relationship/sex (0.65) and relationship/marital-status (0.49): strong overlap
- race/native-country (0.41): moderate association
- Strongest predictors of income: marital-status, relationship, education, occupation

## Outliers

In [ ]:
def count_outliers_iqr(col):
    Q1 = col.quantile(0.25)
    Q3 = col.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = col[(col < lower) | (col > upper)]
    return len(outliers), round(len(outliers) / len(col) * 100, 2), lower, upper

In [ ]:
outlier_summary = pd.DataFrame([
    {"feature": col,
     "outlier_count": count_outliers_iqr(df[col])[0],
     "outlier_percent": count_outliers_iqr(df[col])[1],
     "lower_bound": count_outliers_iqr(df[col])[2],
     "upper_bound": count_outliers_iqr(df[col])[3]}
    for col in numerical_cols
])

print(outlier_summary.to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.boxplot(x=df[col], ax=axes[i], color="steelblue")
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    axes[i].axvline(lower, color="red", linestyle="--", linewidth=0.8, label="IQR bounds")
    axes[i].axvline(upper, color="red", linestyle="--", linewidth=0.8)
    axes[i].set_title(col)
    axes[i].set_xlabel("")

plt.suptitle("Outliers by Feature (IQR Method)", y=1.02)
plt.tight_layout()
plt.show()

Findings:

- hours-per-week: high number of flagged values clustered near the IQR boundary,
  suggesting a heavy-tailed distribution rather than isolated anomalies.
  These are not considered candidates for removal.

- capital-gain: one extreme value (~100k) identified as a clear statistical outlier.
  Given the zero-inflated nature of this feature, this single value has limited
  impact on the overall distribution and is retained.

- capital-loss: few flagged values, no action required.

- General: no outliers are removed. In a census context, flagged values may 
  represent plausible real-world observations rather than data errors.

## Summary of Findings

### Dataset
- 48,842 observations, 14 features, 1 binary target (`income`)
- Mix of 6 numerical and 8 categorical features
- Source: 1994 US Census data

### Data Quality
- Missing values encoded as `?` in three features: `workclass` (~5.6%), 
  `occupation` (~5.7%), `native-country` (~1.8%)
- Target labels contained formatting artifacts (trailing dots) — standardized
- `native-country` contains an ambiguous category `South` that cannot be 
  resolved without original census documentation

### Target Distribution
- Moderately imbalanced: ~76% `<=50K`, ~24% `>50K`
- Raw accuracy may be a misleading evaluation metric given the class imbalance, F1 and AUC-ROC are considered more appropriate in this context

### Numerical Features
- No strong linear correlations between numerical features (all < 0.2)
- `capital-gain` and `capital-loss` are heavily zero-inflated with long right tails
- `fnlwgt` shows near-zero correlation with all other features and the target
- `education-num` takes only ~16 discrete values — better treated as ordinal
- No outlier removal applied. Flagged values represent plausible census observations

### Categorical Features
- `education` and `education-num` encode redundant information
- `relationship`, `marital-status` and `sex` show strong inter-feature associations 
  (Cramér's V up to 0.65) suggesting overlapping information
- `race` and `native-country` are heavily imbalanced toward `White` and 
  `United-States` respectively
- Rare categories in `workclass`, `occupation` and `native-country` may be 
  challenging for generative models to reproduce faithfully

### Target Associations
- Strongest associating features with `income`: `marital-status` (0.45), 
  `relationship` (0.45), `education` (0.37), `occupation` (0.35)
- `sex` shows a moderate association (0.21) with notable income distribution differences

### Implications for Synthetic Data Generation
- Zero-inflated features (`capital-gain`, `capital-loss`) pose a known challenge 
  for generative models
- Class imbalance, rare categories and inter-feature dependencies should be 
  preserved in synthetic data — deviations are measurable utility loss
- Fairness-relevant features (`race`, `sex`) require particular attention — 
  distortion in synthetic data could introduce or amplify bias

In [ ]:
summary_flags = pd.DataFrame([
    {"feature": "capital-gain",    "finding": "zero-inflated, long tail",          "implication": "challenging for synthesis"},
    {"feature": "capital-loss",    "finding": "zero-inflated, moderate outliers",   "implication": "challenging for synthesis"},
    {"feature": "fnlwgt",          "finding": "near-zero target correlation",       "implication": "candidate for dropping"},
    {"feature": "education-num",   "finding": "redundant with education",           "implication": "consider dropping one"},
    {"feature": "relationship",    "finding": "strong overlap with sex/marital",    "implication": "potential redundancy"},
    {"feature": "race",            "finding": "heavily imbalanced",                 "implication": "fairness risk in synthesis"},
    {"feature": "native-country",  "finding": "dominated by United-States, ambiguous South", "implication": "rare categories at risk"},
    {"feature": "income",          "finding": "76/24 class imbalance",              "implication": "use F1/AUC for evaluation"},
])

summary_flags